<a href="https://colab.research.google.com/github/chunchubalaramakrishna/Gen-AI-ASR-Synthetic-Data-medical-dialogue-healthcare/blob/main/Physician_Patient_ASR_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Install the ElevenLabs Python library
!pip install -q elevenlabs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.4 MB/s eta 0:00:00


In [8]:
from elevenlabs import Voice, VoiceSettings
from elevenlabs.client import ElevenLabs
from google.colab import userdata
import json
import os
from IPython.display import Audio # Import Audio for playback

# Initialize the ElevenLabs client
client = None # Initialize client to None
try:
    ELEVENLABS_API_KEY = userdata.get('ELEVENLABS_API_KEY')
    client = ElevenLabs(api_key=ELEVENLABS_API_KEY) # Pass API key to constructor
    print("ElevenLabs API key loaded and client initialized successfully.")
except userdata.SecretError:
    print("Error: ELEVENLABS_API_KEY not found in Colab secrets. Please add it.")
except Exception as e:
    print(f"An error occurred while initializing ElevenLabs client: {e}")

ElevenLabs API key loaded and client initialized successfully.


In [9]:
# Define Voice Settings and IDs
def create_voice_settings_from_dict(settings_dict):
    return VoiceSettings(
        stability=settings_dict.get("stability", 0.5),
        similarity_boost=settings_dict.get("similarity_boost", 0.5),
        style=settings_dict.get("style", 0.0),
        use_speaker_boost=settings_dict.get("use_speaker_boost", True)
    )

# Recommended Voice Settings from the prompt
physician_settings_dict = {
  "stability": 0.70,
  "similarity_boost": 0.85,
  "style": 0.25
}

patient_settings_dict = {
  "stability": 0.55,
  "similarity_boost": 0.80,
  "style": 0.35
}

physician_voice_settings = create_voice_settings_from_dict(physician_settings_dict)
patient_voice_settings = create_voice_settings_from_dict(patient_settings_dict)

print("Physician Voice Settings:", physician_voice_settings)
print("Patient Voice Settings:", patient_voice_settings)

# Fetch all available voices to get their IDs
print("Fetching ElevenLabs voice IDs...")
available_voices = client.voices.get_all()

PHYSICIAN_VOICE_ID = None
PATIENT_VOICE_ID = None

# Define the target names for matching (using substrings for flexibility)
PHYSICIAN_VOICE_NAME_TARGET = "Adam"
PATIENT_VOICE_NAME_TARGET = "Bella" # 'Rachel' was not found, using 'Bella' as an alternative.

found_physician_voice = False
found_patient_voice = False

print("\n--- Available ElevenLabs Voices ---")
for voice in available_voices.voices:
    print(f"Name: {voice.name}, ID: {voice.voice_id}, Category: {voice.category}")
    if PHYSICIAN_VOICE_NAME_TARGET in voice.name:
        PHYSICIAN_VOICE_ID = voice.voice_id
        found_physician_voice = True
    if PATIENT_VOICE_NAME_TARGET in voice.name:
        PATIENT_VOICE_ID = voice.voice_id
        found_patient_voice = True
print("---------------------------------\n")

if PHYSICIAN_VOICE_ID:
    print(f"Found Physician voice ('{PHYSICIAN_VOICE_NAME_TARGET}') with ID: {PHYSICIAN_VOICE_ID}")
else:
    print(f"Error: Physician voice '{PHYSICIAN_VOICE_NAME_TARGET}' not found among available voices. Please check ElevenLabs account or API key permissions. If '{PHYSICIAN_VOICE_NAME_TARGET}' is a custom voice, ensure it's created in your account.")

if PATIENT_VOICE_ID:
    print(f"Found Patient voice ('{PATIENT_VOICE_NAME_TARGET}') with ID: {PATIENT_VOICE_ID}")
else:
    print(f"Error: Patient voice '{PATIENT_VOICE_NAME_TARGET}' not found among available voices. Please check ElevenLabs account or API key permissions. If '{PATIENT_VOICE_NAME_TARGET}' is a custom voice, ensure it's created in your account.")

if not PHYSICIAN_VOICE_ID or not PATIENT_VOICE_ID:
    raise ValueError("One or both required voice IDs not found. Audio generation cannot proceed. Please select available voices from the list above or ensure they are configured in your ElevenLabs account.")

Physician Voice Settings: stability=0.7 use_speaker_boost=True similarity_boost=0.85 style=0.25 speed=None
Patient Voice Settings: stability=0.55 use_speaker_boost=True similarity_boost=0.8 style=0.35 speed=None
Fetching ElevenLabs voice IDs...

--- Available ElevenLabs Voices ---
Name: Roger - Laid-Back, Casual, Resonant, ID: CwhRBWXzGAHq8TQ4Fs17, Category: premade
Name: Sarah - Mature, Reassuring, Confident, ID: EXAVITQu4vr4xnSDxMaL, Category: premade
Name: Laura - Enthusiast, Quirky Attitude, ID: FGY2WhTYpPnrIDTdsKH5, Category: premade
Name: Charlie - Deep, Confident, Energetic, ID: IKne3meq5aSn9XLyUdCD, Category: premade
Name: George - Warm, Captivating Storyteller, ID: JBFqnCBsd6RMkjVDRZzb, Category: premade
Name: Callum - Husky Trickster, ID: N2lVS1w4EtoT3dr4eOWO, Category: premade
Name: River - Relaxed, Neutral, Informative, ID: SAz9YHcvj6GT2YYXdXww, Category: premade
Name: Harry - Fierce Warrior, ID: SOYHLrjzK2X1ezoPC6cr, Category: premade
Name: Liam - Energetic, Social Media C

In [10]:
#Medical Terminology
medical_terms = [
    "HbA1c",
    "Metformin",
    "Metformin XR",
    "Lisinopril"
]

# Example:
physician_text_template = "Hello, Bella. Your recent {term1} test results show improvement. We'll continue with {term2} and monitor your blood pressure, which is currently managed by {term3}."
patient_text_template = "Hello Doctor Adam, I have a question about {term1} and how it affects my {term2}."


In [11]:
#Generate Speech Samples
def generate_and_play_audio(text, voice_id_str, voice_settings, filename=None):
    print(f"Generating audio for voice ID '{voice_id_str}': {text[:70]}...\n")

    audio_response = client.text_to_speech.convert(
        text=text,
        voice_id=voice_id_str,
        voice_settings=voice_settings,
        model_id="eleven_multilingual_v2"
    )

    audio_bytes = b''.join(list(audio_response))

    display(Audio(audio_bytes, rate=16000))

    if filename:
        os.makedirs(os.path.dirname(filename) or '.', exist_ok=True)
        with open(filename, 'wb') as f:
            f.write(audio_bytes)
        print(f"Audio saved to {filename}")
    print("\n")
    return audio_bytes

# Physician's dialogue
physician_dialogue = physician_text_template.format(
    term1=medical_terms[0],
    term2=medical_terms[2],
    term3=medical_terms[3]
)
physician_audio = generate_and_play_audio(physician_dialogue, PHYSICIAN_VOICE_ID, physician_voice_settings, "physician_dialogue.wav")

# Patient's dialogue
patient_dialogue = patient_text_template.format(
    term1=medical_terms[1],
    term2="treatment plan"
)
patient_audio = generate_and_play_audio(patient_dialogue, PATIENT_VOICE_ID, patient_voice_settings, "patient_dialogue.wav")

Generating audio for voice ID 'pNInz6obpgDQGcFmaJgB': Hello, Bella. Your recent HbA1c test results show improvement. We'll c...



Audio saved to physician_dialogue.wav


Generating audio for voice ID 'hpp4J3VqNfWAUOO0d1Us': Hello Doctor Adam, I have a question about Metformin and how it affect...



Audio saved to patient_dialogue.wav




#Dataset Metadata Example

```json
{
  "vertical": "Healthcare",
  "scenario": "Telehealth Consultation",
  "sample_rate": 16000,
  "speakers": 2,
  "speaker_a_role": "Physician",
  "speaker_b_role": "Patient",
  "medical_terms": [
    "HbA1c",
    "Metformin",
    "Metformin XR",
    "Lisinopril"
  ],
  "audio_file": "physician_patient_dialogue_1.wav"
}
```
